# 03 — Job-density model (Phase 3)

**The core challenge:** official job counts exist only as 8 district totals
(payroll jobs, counted at the workplace). That is far too few for ordinary
supervised ML, so both models build a *weight surface* over hexes and
rescale it so each district's hex sum equals the official total
(mass-preserving dasymetric disaggregation, the WorldPop approach):

- **Model A** — GFA × coefficient heuristic: transparent, the benchmark.
- **Model B** — random forest trained on log job density at district level
  (8 districts × 5 years), predicting per hex.

Validation is **leave-one-district-out** (LODO): train on 7 districts,
predict the 8th's total, compare with the official number. By default this
notebook runs on the committed sample (one district → the forest is skipped
with a recorded reason); set `ALMATY_CONFIG=config/config.yaml` for the city.

In [ ]:
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from almaty_access.config import load_config
from almaty_access.model.build import MODEL_REPORT_FILE, build_model

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
CONFIG_PATH = Path(os.environ.get("ALMATY_CONFIG", REPO / "config" / "config.sample.yaml"))
config = load_config(CONFIG_PATH)
build_model(config)  # cached

P = config.paths.processed
jobs8 = pd.read_parquet(P / "jobs_r8.parquet")
jobs9 = pd.read_parquet(P / "jobs_r9.parquet")
grid9 = gpd.read_parquet(P / "grid_r9.parquet")
report = json.loads((P / MODEL_REPORT_FILE).read_text("utf-8"))
RF_ACTIVE = not report["rf_skipped_reason"]
print(
    f"config: {CONFIG_PATH.name} | target year {report['target_year']} "
    f"({report['target_indicator']}, training years {report['training_years']})"
)
print(f"districts modeled: {report['districts_modeled']}")
if not RF_ACTIVE:
    print("NOTE:", report["rf_skipped_reason"])
print(f"total jobs placed: {jobs8['jobs_rf'].sum():,.0f}")

## Leave-one-district-out: does Model B beat Model A?

Read this table honestly. On the full city the forest wins decisively on
**MAPE** (relative error — the fair metric when districts span 29k–255k
jobs), driven by far better predictions for the small dormitory districts.
The baseline keeps a lower **MAE** because the forest underpredicts
held-out Almaly: trees cannot extrapolate beyond their training range, and
with the densest district held out nothing in training looks like it —
a textbook RF limitation worth naming in any interview. The production
surface never relies on predicted *totals* (official numbers pin them);
LODO measures how well the learned density *shape* generalises.

In [ ]:
if RF_ACTIVE:
    lodo = pd.DataFrame(report["lodo"])
    metrics = pd.DataFrame(
        {"baseline": report["lodo_metrics_baseline"], "random forest": report["lodo_metrics_rf"]}
    ).T
    print(metrics.round(3).to_string())
    print("\nModel B beats Model A on MAPE:", report["rf_beats_baseline"])
    ax = lodo.set_index("district")[["actual", "pred_baseline", "pred_rf"]].plot.bar(
        figsize=(10, 4), color=["#444444", "#9ecae1", "#fd8d3c"]
    )
    ax.set_ylabel("jobs")
    ax.set_title("LODO: actual vs predicted district totals")
    plt.tight_layout()
else:
    print("LODO skipped:", report["rf_skipped_reason"])

## Where are the jobs? Baseline vs forest, and the uncertainty band

Both maps conserve the same district totals; the difference is the
within-district shape. The third panel shows the relative forest spread
(P90 − P10) / (P50 + 1): wide where trees disagree — treat those hexes
with care in Phases 4–5.

In [ ]:
joined = grid9.merge(jobs9, on="h3", suffixes=("", "_j")).to_crs(config.crs.metric)
joined["spread"] = (joined["jobs_rf_p90"] - joined["jobs_rf_p10"]) / (joined["jobs_rf"] + 1)
panels = [
    ("jobs_baseline", "Model A (log1p)"),
    ("jobs_rf", "Model B (log1p)"),
    ("spread", "relative uncertainty"),
]
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, (column, title) in zip(axes, panels, strict=True):
    values = np.log1p(joined[column]) if column.startswith("jobs") else joined[column]
    joined.assign(v=values).plot(
        ax=ax, column="v", cmap="magma", legend=True, legend_kwds={"shrink": 0.6}
    )
    ax.set_title(title)
    ax.set_axis_off()
fig.tight_layout()

## Plausibility: top hexes, ratios, industry and anchors

README §3.3's checks: the top-20 hexes should be recognisable business
areas (on the city: the Almaly core, Bostandyk around Al-Farabi/Esentai,
bazaar and industrial corridors); the jobs-to-residents ratio should show
the centre attracting (≫1) and dormitory districts producing (≪1);
industrial-GFA hexes should combine high jobs with low population; and
hexes holding hospitals or universities should rank above the median hex
(the report's `plausibility` block carries the numbers).

In [ ]:
top = pd.DataFrame(report["top20_hexes_rf"])
print(top.to_string(index=False))
print("\nplausibility (README §3.3):")
for key, value in report["plausibility"].items():
    print(f"  {key}: {value}")
ratios = pd.DataFrame(report["jobs_per_resident"]).T
ax = ratios["jobs_per_resident"].sort_values().plot.barh(figsize=(7, 3.5), color="#30808f")
ax.axvline(1.0, color="grey", linestyle="--")
ax.set_title("Official jobs per resident (attraction vs dormitory districts)")
plt.tight_layout()

## Why does the forest place jobs where it does?

Permutation importance (computed in the pipeline, district level) plus
SHAP values on a refit of the same seeded forest. With 40 training rows
and collinear features the credit is spread thin — read the *group* of top
features (business POIs, commercial/large GFA), not single rankings.

In [ ]:
if RF_ACTIVE:
    importances = pd.DataFrame(report["permutation_importance_top15"])
    ax = (
        importances.set_index("feature")["importance"]
        .iloc[::-1]
        .plot.barh(figsize=(8, 5), color="#30808f")
    )
    ax.set_title("Permutation importance (district level, top 15)")
    plt.tight_layout()
else:
    print("skipped:", report["rf_skipped_reason"])

In [ ]:
if RF_ACTIVE:
    import shap

    from almaty_access.model.aggregate import apply_log1p, district_design
    from almaty_access.model.forest import fit_forest, training_table
    from almaty_access.model.target import load_district_targets, resolve_years

    features8 = pd.read_parquet(P / "features_r8.parquet")
    targets = load_district_targets(config)
    _, training_years = resolve_years(targets, config)
    design = apply_log1p(district_design(features8, config), config.features.log1p_prefixes)
    X, y, _ = training_table(design, targets, training_years)
    forest = fit_forest(X, y, config)  # same seed -> same forest as the pipeline
    explainer = shap.TreeExplainer(forest)
    shap_values = explainer.shap_values(X)
    shap.summary_plot(shap_values, X, max_display=12, show=False)
    plt.title("SHAP: what drives predicted log job density")
    plt.tight_layout()
else:
    print("skipped:", report["rf_skipped_reason"])

## Sensitivity (README §3.4): coefficients, building heights, POI mappings

Three axes, from cheapest to most structural:

1. **Model A coefficients** — rescaling absorbs absolute levels, so only
   ratios matter; doubling/halving the commercial coefficient should
   re-rank hexes mildly.
2. **Building-height assumptions** — the critical axis here: **97% of
   floor counts come from the config `default_levels`** (OSM levels tags
   are sparse), so the GFA surface both models consume largely rests on
   those defaults. We rebuild per-hex GFA from `buildings.parquet` under
   varied defaults and re-run both models.
3. **POI category mappings** — Model B's top features are POI counts; we
   merge an ambiguous category (`retail_other` → `grocery`) and refit.

Approximation, stated: the variant runs swap the GFA / POI-count columns
the models read directly; derived columns (large-building share, k-ring
context) stay at shipped values, and variant buildings are assigned by
representative point. Rank stability (Spearman ρ vs the shipped surface)
is the metric — a ρ far below 1 would mean the assumption, not the data,
drives the result.

In [ ]:
import dataclasses

from scipy.stats import spearmanr

from almaty_access.model.baseline import baseline_weights, rescale_to_totals
from almaty_access.model.target import load_district_targets, resolve_years

features9 = pd.read_parquet(P / "features_r9.parquet")
targets_df = load_district_targets(config)
totals9 = {
    d: float(
        targets_df.loc[
            (targets_df["district"] == d) & (targets_df["year"] == report["target_year"]),
            "jobs",
        ].iloc[0]
    )
    for d in report["districts_modeled"]["r9"]
}
district_of9 = features9.set_index("h3")["district"]
base9 = rescale_to_totals(baseline_weights(features9, config), district_of9, totals9)
rows = []
for factor in (0.5, 2.0):
    coefficients = dict(config.model.baseline_jobs_per_100m2)
    coefficients["commercial"] = coefficients["commercial"] * factor
    variant_config = dataclasses.replace(
        config, model=dataclasses.replace(config.model, baseline_jobs_per_100m2=coefficients)
    )
    variant = rescale_to_totals(baseline_weights(features9, variant_config), district_of9, totals9)
    occupied = (base9 > 0) | (variant > 0)
    rho = spearmanr(base9[occupied], variant[occupied]).statistic
    rows.append(
        {
            "axis": "A coefficients",
            "variant": f"commercial x{factor}",
            "spearman_vs_shipped": round(float(rho), 3),
        }
    )
sensitivity_rows = rows
pd.DataFrame(rows)

In [ ]:
import h3 as h3lib

from almaty_access.io import read_geoparquet
from almaty_access.model.aggregate import apply_log1p, district_design, hex_design
from almaty_access.model.build import _job_capable, _rescaled_tree_surfaces
from almaty_access.model.forest import fit_forest, training_table, tree_weights
from almaty_access.schemas import BUILDING_FUNCTIONS

res8 = config.h3.analysis_resolution
features8 = pd.read_parquet(P / "features_r8.parquet")
district_of8 = features8.set_index("h3")["district"]
totals8 = {
    d: float(
        targets_df.loc[
            (targets_df["district"] == d) & (targets_df["year"] == report["target_year"]),
            "jobs",
        ].iloc[0]
    )
    for d in report["districts_modeled"]["r8"]
}
shipped_base8 = jobs8.set_index("h3")["jobs_baseline"]
shipped_rf8 = jobs8.set_index("h3")["jobs_rf"]
_, training_years = resolve_years(targets_df, config)


def rf_surface(features_variant: pd.DataFrame) -> pd.Series:
    """Refit the seeded forest on a variant feature table, r8 surface."""
    prefixes = config.features.log1p_prefixes
    design_d = apply_log1p(district_design(features_variant, config), prefixes)
    X, y, _ = training_table(design_d, targets_df, training_years)
    forest = fit_forest(X, y, config)
    design_h = apply_log1p(hex_design(features_variant), prefixes)[list(X.columns)]
    capable = _job_capable(features_variant).loc[design_h.index].to_numpy()
    surfaces = _rescaled_tree_surfaces(
        tree_weights(forest, design_h) * capable,
        district_of8.loc[design_h.index],
        totals8,
    )
    median = pd.Series(np.quantile(surfaces, 0.5, axis=0), index=design_h.index)
    return rescale_to_totals(median, district_of8.loc[design_h.index], totals8)


def compare(label: str, variant: pd.Series, shipped: pd.Series) -> None:
    occupied = (shipped.reindex(variant.index).fillna(0) > 0) | (variant > 0)
    rho = spearmanr(shipped.reindex(variant.index).fillna(0)[occupied], variant[occupied]).statistic
    sensitivity_rows.append(
        {
            "axis": label.split(":")[0],
            "variant": label.split(":", 1)[1].strip(),
            "spearman_vs_shipped": round(float(rho), 3),
        }
    )


# ---- Axis 2: building-height defaults (97% of floor counts) ----------
buildings = read_geoparquet(config.paths.interim / "buildings.parquet")
points = buildings.geometry.representative_point()
cells8 = pd.Series(
    [h3lib.latlng_to_cell(lat, lon, res8) for lat, lon in zip(points.y, points.x, strict=True)],
    index=buildings.index,
)
height_variants = {
    "commercial/public 2 -> 3 floors": {
        **config.buildings.default_levels,
        "commercial": 3.0,
        "public": 3.0,
    },
    "all defaults x0.75": {k: v * 0.75 for k, v in config.buildings.default_levels.items()},
}
for label, default_levels in height_variants.items():
    levels = buildings["levels"].where(
        buildings["levels_source"] != "default",
        buildings["function"].map(default_levels),
    )
    gfa = pd.DataFrame(
        {"h3": cells8, "function": buildings["function"], "gfa": buildings["footprint_m2"] * levels}
    )
    pivot = gfa.pivot_table(index="h3", columns="function", values="gfa", aggfunc="sum")
    features_v = features8.copy()
    indexed = features_v.set_index("h3")
    for function in BUILDING_FUNCTIONS:
        column = pivot[function] if function in pivot.columns else pd.Series(dtype=float)
        indexed[f"bf_gfa_{function}_m2"] = column.reindex(indexed.index).fillna(0.0)
    indexed["bf_gfa_m2"] = sum(indexed[f"bf_gfa_{f}_m2"] for f in BUILDING_FUNCTIONS)
    features_v = indexed.reset_index()
    base_v = rescale_to_totals(baseline_weights(features_v, config), district_of8, totals8)
    compare(f"B heights: {label} (Model A)", base_v, shipped_base8)
    if RF_ACTIVE:
        compare(f"B heights: {label} (Model B)", rf_surface(features_v), shipped_rf8)

# ---- Axis 3: POI mapping perturbation ---------------------------------
if RF_ACTIVE:
    features_p = features8.copy()
    features_p["poi_cnt_grocery"] = (
        features_p["poi_cnt_grocery"] + features_p["poi_cnt_retail_other"]
    )
    features_p["poi_cnt_retail_other"] = 0
    compare("C poi-mapping: retail_other -> grocery (Model B)", rf_surface(features_p), shipped_rf8)
else:
    print("Model B axes skipped:", report["rf_skipped_reason"])

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity

## Verdict

In [ ]:
conservation_ok = True
for district in report["districts_modeled"]["r9"]:
    mask = jobs9["district"].eq(district).fillna(False)
    official = report["official_totals"][district]
    # The report rounds totals to whole jobs; conservation itself is
    # exact (test_model_pipeline checks it against the raw targets).
    conservation_ok &= abs(jobs9.loc[mask, "jobs_rf"].sum() - official) / official < 1e-4
checks = {
    "district totals conserved (rf, r9)": bool(conservation_ok),
    "uncertainty band ordered (p10 <= p90)": bool(
        (jobs9["jobs_rf_p10"] <= jobs9["jobs_rf_p90"] + 1e-9).all()
    ),
    "buffer hexes carry zero jobs": bool(
        (jobs9.loc[~jobs9["in_city"], ["jobs_baseline", "jobs_rf"]] == 0).all().all()
    ),
    "top-20 hexes are city hexes": bool(
        pd.DataFrame(report["top20_hexes_rf"])["district"].notna().all()
    ),
    "sensitivity covers all feasible axes": bool(len(sensitivity) >= (7 if RF_ACTIVE else 4)),
    # Full-city criteria; skipped (None) when the forest was skipped.
    "Model B beats Model A on LODO MAPE": report["rf_beats_baseline"],
    "top features are business/built-form signals": (
        None
        if not RF_ACTIVE
        else bool(
            pd.DataFrame(report["permutation_importance_top15"])["feature"]
            .str.startswith(("poi_", "bf_", "ctx_", "pop_", "net_"))
            .mean()
            > 0.6
        )
    ),
}
for name, ok in checks.items():
    print(("SKIP " if ok is None else "PASS " if ok else "FAIL ") + name)
print(f"\nleast rank-stable sensitivity variant: rho = {sensitivity['spearman_vs_shipped'].min()}")
assert all(ok for ok in checks.values() if ok is not None), "Phase 3 'done when' not satisfied"
print("Job surfaces are ready for Phase 4 accessibility.")